# 03 — KnottedGraph vs Topoly: focused paper scaling

This notebook now runs only the **two benchmark families used in the paper**:

1. **Crossing complexity at fixed graph size:** a single theta graph with $V=2$ and $E=3$, while only the projected crossing count $c$ increases.
2. **Trivalent input-size scaling:** planar $K_4$ components, plotted against total vertex count $V$.

The previously generated throughput, edge-theta, prism-$V$, prism-$E$, and random-cubic figures are intentionally not executed here. This keeps the long local run focused on the two paper figures and avoids redundant Yamada evaluations.

For every sample, graph construction and PD-code construction happen **outside** the timed region. KnottedGraph and Topoly receive the same PD code. Each x-value uses independent deterministic embeddings; the plotted center is the median across embeddings with a nonparametric 95% bootstrap confidence interval.

A completed long run is cached locally. Re-running the notebook with unchanged code/configuration loads the cached sample rows and proceeds directly to validation and plotting.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, subprocess, sys

from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import knotted_graph

kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Configuration

A normal local run uses the paper profile:

- 10 independent deterministic embeddings per x-value;
- one timed Yamada evaluation per framework per embedding;
- up to 120 s wall time per framework/sample;
- automatic censor-frontier stopping after repeated fully timed-out x-values.

Using exactly one timed evaluation per embedding avoids the old change in stopwatch-repeat count around particular crossing values. Statistical replication comes from the independent embeddings themselves.

GitHub Actions uses a small smoke profile. Do not use smoke-profile timing values in the paper.


In [ ]:
IS_CI = os.environ.get("CI", "").lower() == "true"

PROFILE = "smoke" if IS_CI else "paper"
SAMPLES_PER_X = 2 if IS_CI else 10
TIMEOUT_S = 10 if IS_CI else 120
BASE_SEED = 20260818
CENSOR_FRONTIER = 2

raw_csv = RES / "topoly_yamada_paper_scaling_raw.csv"
aggregate_csv = RES / "topoly_yamada_paper_scaling_aggregate.csv"

print(
    f"mode={PROFILE}, samples/x={SAMPLES_PER_X}, "
    f"timeout/framework/sample={TIMEOUT_S}s"
)


In [ ]:
def _load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def run_streamed_benchmark(cmd, *, total, description):
    print("Running:", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Benchmark subprocess did not expose stdout.")

    streamed_rows = []
    summary_rows = None
    bar = tqdm(total=total, desc=description, unit="sample", dynamic_ncols=True)

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue

        if line.startswith("SUMMARY="):
            summary_rows = json.loads(line[len("SUMMARY="):])
            continue

        if line.startswith("{"):
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                tqdm.write(line)
                continue
            if "family" in row:
                streamed_rows.append(row)
                meta = " ".join(
                    f"{key}={row[key]}"
                    for key in ("V", "E", "crossings")
                    if row.get(key) is not None
                )
                bar.set_postfix_str(
                    f"{row.get('family')} {meta} "
                    f"sample={row.get('embedding','?')} "
                    f"KG={row.get('knottedgraph_status','?')},"
                    f"T={row.get('topoly_status','?')}"
                )
                bar.update(1)
                continue

        tqdm.write(line)

    return_code = process.wait()
    bar.close()
    if return_code:
        raise RuntimeError(
            f"Benchmark failed with exit code {return_code}: "
            f"{' '.join(map(str, cmd))}"
        )

    rows = summary_rows if summary_rows is not None else streamed_rows
    if not rows:
        raise RuntimeError("Benchmark completed without sample rows.")
    print(f"completed {len(rows)} sample records")
    return rows


## 2. Run only the two paper benchmark families

The driver below contains only:

- `crossings_fixed`: fixed $V=2$, fixed $E=3$, varying $c$;
- `vertices_k4`: planar trivalent $K_4$ components, varying total $V$.

No throughput, edge-theta, prism, or random-cubic benchmark is run by this notebook.


In [ ]:
paper_script = ROOT / "dev" / "benchmark_topoly_paper_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
env["PYTHONNOUSERSITE"] = "1"

paper_module = _load_module(
    paper_script,
    "kg_topoly_paper_scaling_notebook_plan",
)
plan = paper_module.paper_cases(PROFILE)

assert set(plan) == {"crossings_fixed", "vertices_k4"}
total = sum(len(cases) for cases in plan.values()) * SAMPLES_PER_X

cmd = [
    sys.executable,
    str(paper_script),
    "--profile", PROFILE,
    "--embeddings", str(SAMPLES_PER_X),
    "--timeout", str(TIMEOUT_S),
    "--seed", str(BASE_SEED),
    "--censor-frontier", str(CENSOR_FRONTIER),
]

rows = run_streamed_benchmark(
    cmd,
    total=total,
    description="Paper Yamada scaling",
)


## 3. Acceptance checks and raw-data export

Before plotting, the notebook verifies that the raw records contain only the two intended families and satisfy their construction constraints.

For `crossings_fixed` every sample must satisfy

$$
V=2,\qquad E=3.
$$

For `vertices_k4`, the requested size equals the actual total vertex count $V$ and every sample has zero projected crossings by construction.

Whenever both frameworks finish successfully, their Laurent polynomials must agree up to the already-validated Yamada convention conversion.


In [ ]:
from collections import defaultdict

assert {row["family"] for row in rows} <= {
    "crossings_fixed",
    "vertices_k4",
}

groups = defaultdict(list)
for row in rows:
    groups[(row["family"], int(row["size"]))].append(row)

for key, group in groups.items():
    assert len(group) == SAMPLES_PER_X, (key, len(group), SAMPLES_PER_X)
    assert len({row["embedding"] for row in group}) == SAMPLES_PER_X
    assert len({row["embedding_hash"] for row in group}) == SAMPLES_PER_X

for row in rows:
    if row["family"] == "crossings_fixed":
        assert int(row["V"]) == 2
        assert int(row["E"]) == 3
        assert int(row["crossings"]) == int(row["size"])
    elif row["family"] == "vertices_k4":
        assert int(row["V"]) == int(row["size"])
        assert int(row["crossings"]) == 0

    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == "ok"
        assert row["topoly_status"] == "ok"
        assert row["pd_hash"]

keys = list(dict.fromkeys(key for row in rows for key in row))
with raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)

print(f"PASS: only the two paper benchmark families were evaluated.")
print(f"PASS: each x point contains {SAMPLES_PER_X} distinct embeddings.")
print(f"wrote {len(rows)} sample-level records to {raw_csv}")


## 4. Generate the two paper figures

Only two figure pairs are expected:

1. `topoly_vs_knottedgraph_crossings_fixed.{png,pdf}`
2. `topoly_vs_knottedgraph_vertices_k4.{png,pdf}`

Because the raw CSV contains no other benchmark families, the plotting utility skips every other historical specification.


In [ ]:
plot_script = ROOT / "dev" / "plot_topoly_scaling.py"
plot_cmd = [
    sys.executable,
    str(plot_script),
    str(raw_csv),
    "--figure-dir", str(FIG),
    "--aggregate-csv", str(aggregate_csv),
]

print("Running:", " ".join(plot_cmd))
plot_process = subprocess.Popen(
    plot_cmd,
    cwd=ROOT,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
if plot_process.stdout is None:
    raise RuntimeError("Plot subprocess did not expose stdout.")

for line in plot_process.stdout:
    print(line, end="")

if plot_process.wait():
    raise RuntimeError("Paper-quality plot generation failed.")

expected_stems = [
    "topoly_vs_knottedgraph_crossings_fixed",
    "topoly_vs_knottedgraph_vertices_k4",
]
for stem in expected_stems:
    assert (FIG / f"{stem}.png").exists(), stem
    assert (FIG / f"{stem}.pdf").exists(), stem

assert aggregate_csv.exists()
print("PASS: both paper figure PNG/PDF pairs and aggregate CSV were created.")


## 5. Interpretation

The two retained figures answer complementary questions.

**Crossing-complexity figure.** The graph size is fixed at $V=2$, $E=3$, so the x-axis isolates increasing diagram complexity through the projected crossing count $c$.

**Input-size figure.** The benchmark increases the total number of vertices through planar trivalent $K_4$ components. This probes graph/input-size scaling in a simple controlled family.

The benchmark does not time graph construction or PD-code construction. It times only the Yamada evaluator after the common PD input has already been prepared for both frameworks.

Timeouts are treated as censored observations rather than substituted into the bootstrap confidence interval.
